In [1]:
import os
import re
from pathlib import Path
import pandas as pd
from PIL import Image

# Define project base directories
PROJECT_ROOT = Path(r"E:\Python Projects\Celebrity_Face_matcher")
STAGING_DIR = PROJECT_ROOT / "data" / "staging"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Ensure metadata and processed directories exist
METADATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

BOLLYWOOD_ROOT = STAGING_DIR / "Bollywood_celeb_face_localized"
PINS_ROOT = STAGING_DIR / "105_classes_pins_dataset"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Bollywood Staging Exists: {BOLLYWOOD_ROOT.exists()}")
print(f"Pins Staging Exists: {PINS_ROOT.exists()}")

Project Root: E:\Python Projects\Celebrity_Face_matcher
Bollywood Staging Exists: True
Pins Staging Exists: True


In [2]:
VALID_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def count_valid_images(folder_path: Path):
    return sum(1 for p in folder_path.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS)

def format_canonical_name(name_str: str) -> str:
    # Replace underscores/hyphens with space and format to Title Case
    cleaned = re.sub(r"[_\-]+", " ", name_str).strip()
    return " ".join(word.capitalize() for word in cleaned.split())

def to_dir_slug(canonical_name: str) -> str:
    # Safe directory name: lowercase with underscores
    return re.sub(r"\s+", "_", canonical_name.lower())

registry_records = []

# --- 1. Audit Bollywood Dataset ---
bolly_partitions = [BOLLYWOOD_ROOT / f"bollywood_celeb_faces_{i}" for i in (0, 1)] + [BOLLYWOOD_ROOT / "bollywood_celeb_faces2"]

for partition in bolly_partitions:
    if not partition.exists():
        print(f"Warning: Partition not found: {partition}")
        continue
    for actor_dir in sorted(partition.iterdir()):
        if actor_dir.is_dir():
            canonical = format_canonical_name(actor_dir.name)
            img_count = count_valid_images(actor_dir)
            registry_records.append({
                "canonical_name": canonical,
                "folder_slug": to_dir_slug(canonical),
                "domain": "Bollywood",
                "sub_domain": "Cinema",
                "raw_staging_path": str(actor_dir),
                "raw_image_count": img_count
            })

# --- 2. Audit Pins Hollywood / Global Dataset ---
for celeb_dir in sorted(PINS_ROOT.iterdir()):
    if celeb_dir.is_dir():
        raw_name = celeb_dir.name
        # Remove pins_ or pins- prefix
        clean_name = re.sub(r"^pins[_\-\s]+", "", raw_name, flags=re.IGNORECASE)
        canonical = format_canonical_name(clean_name)
        img_count = count_valid_images(celeb_dir)
        registry_records.append({
            "canonical_name": canonical,
            "folder_slug": to_dir_slug(canonical),
            "domain": "Hollywood_Global",
            "sub_domain": "Entertainment_Tech",
            "raw_staging_path": str(celeb_dir),
            "raw_image_count": img_count
        })

df_registry = pd.DataFrame(registry_records)
print(f"Total parsed identities: {len(df_registry)}")

Total parsed identities: 205


In [3]:
# Check if any celebrity is in both datasets (e.g., Priyanka Chopra)
duplicate_names = df_registry[df_registry.duplicated(subset=["folder_slug"], keep=False)]
if not duplicate_names.empty:
    print("Detected cross-dataset identity overlap:")
    print(duplicate_names[["canonical_name", "domain", "raw_image_count"]])
else:
    print("No cross-dataset name collisions detected.")

# Add unique celebrity IDs
df_registry.insert(0, "celebrity_id", [f"celeb_{i+1:04d}" for i in range(len(df_registry))])

# Summary metrics
print("\n--- Domain Breakdown ---")
summary = df_registry.groupby("domain").agg(
    total_identities=("canonical_name", "count"),
    total_images=("raw_image_count", "sum"),
    min_images_per_class=("raw_image_count", "min"),
    avg_images_per_class=("raw_image_count", lambda x: round(x.mean(), 1)),
    max_images_per_class=("raw_image_count", "max")
)
print(summary)

No cross-dataset name collisions detected.

--- Domain Breakdown ---
                  total_identities  total_images  min_images_per_class  \
domain                                                                   
Bollywood                      100          8664                    43   
Hollywood_Global               105         17534                    86   

                  avg_images_per_class  max_images_per_class  
domain                                                        
Bollywood                         86.6                   192  
Hollywood_Global                 167.0                   237  


In [4]:
csv_path = METADATA_DIR / "celebrity_registry.csv"
df_registry.to_csv(csv_path, index=False)
print(f"Registry successfully saved to: {csv_path}")
df_registry.head(10)

Registry successfully saved to: E:\Python Projects\Celebrity_Face_matcher\data\metadata\celebrity_registry.csv


,celebrity_id,canonical_name,folder_slug,domain,sub_domain,raw_staging_path,raw_image_count
0,celeb_0001,Aamir Khan,aamir_khan,Bollywood,Cinema,E:\Python Projects\Celebrity_Face_matcher\data...,192
1,celeb_0002,Abhay Deol,abhay_deol,Bollywood,Cinema,E:\Python Projects\Celebrity_Face_matcher\data...,61
2,celeb_0003,Abhishek Bachchan,abhishek_bachchan,Bollywood,Cinema,E:\Python Projects\Celebrity_Face_matcher\data...,73
3,celeb_0004,Aftab Shivdasani,aftab_shivdasani,Bollywood,Cinema,E:\Python Projects\Celebrity_Face_matcher\data...,68
4,celeb_0005,Aishwarya Rai,aishwarya_rai,Bollywood,Cinema,E:\Python Projects\Celebrity_Face_matcher\data...,125
5,celeb_0006,Ajay Devgn,ajay_devgn,Bollywood,Cinema,E:\Python Projects\Celebrity_Face_matcher\data...,80
6,celeb_0007,Akshay Kumar,akshay_kumar,Bollywood,Cinema,E:\Python Projects\Celebrity_Face_matcher\data...,82
7,celeb_0008,Akshaye Khanna,akshaye_khanna,Bollywood,Cinema,E:\Python Projects\Celebrity_Face_matcher\data...,62
8,celeb_0009,Alia Bhatt,alia_bhatt,Bollywood,Cinema,E:\Python Projects\Celebrity_Face_matcher\data...,125
9,celeb_0010,Ameesha Patel,ameesha_patel,Bollywood,Cinema,E:\Python Projects\Celebrity_Face_matcher\data...,82
